In [ ]:
import sys
from pathlib import Path


ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

import sqlite3
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

from config import DB_PATH, FEATURE_COLS, TARGET_COL, REPORTS_DIR

## 1. Data Overview

In [ ]:
import missingno as msno

con = sqlite3.connect(DB_PATH)

# Sample up to 100k rows for the overview so the missing-value plot renders quickly
df_overview = pd.read_sql_query(
    """
    SELECT
        trip_id, fare_amount, passenger_count, trip_distance,
        pickup_zone_id, dropoff_zone_id, pickup_hour, pickup_dow,
        is_weekend, is_rush_hour, time_of_day_sin, time_of_day_cos,
        haversine_km, distance_bucket, fare_per_mile,
        zone_avg_fare, zone_trip_count, avg_speed_zone
    FROM features
    ORDER BY RANDOM()
    LIMIT 100000
    """,
    con,
)

print(f"Sample shape : {df_overview.shape}")
print(f"\nDtypes:\n{df_overview.dtypes.to_string()}")
print(f"\nMissing counts:\n{df_overview.isnull().sum().to_string()}")
df_overview.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
msno.matrix(df_overview, ax=ax, sparkline=False, fontsize=10)
ax.set_title("Missing-value matrix — features table (100 k sample)", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Target Distribution

In [ ]:
fare = df_overview[TARGET_COL].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Raw distribution
axes[0].hist(fare, bins=80, color=sns.color_palette()[0], edgecolor="white", linewidth=0.4)
axes[0].set_title("fare_amount — raw", fontsize=12)
axes[0].set_xlabel("fare_amount ($)")
axes[0].set_ylabel("count")
axes[0].axvline(fare.median(), color="tomato", linestyle="--", linewidth=1.5, label=f"median ${fare.median():.2f}")
axes[0].legend()

# Log-transformed
log_fare = np.log1p(fare)
axes[1].hist(log_fare, bins=80, color=sns.color_palette()[2], edgecolor="white", linewidth=0.4)
axes[1].set_title("fare_amount — log1 transformed", fontsize=12)
axes[1].set_xlabel("log(1 + fare_amount)")
axes[1].set_ylabel("count")
axes[1].axvline(log_fare.median(), color="tomato", linestyle="--", linewidth=1.5,
                label=f"median {log_fare.median():.3f}")
axes[1].legend()

fig.suptitle("Target Distribution: fare_amount", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness (raw):  {fare.skew():.3f}")
print(f"Skewness (log):  {log_fare.skew():.3f}")

## 3. Zone-Level Fare Analysis (Top Pickup Zones by Volume)

The source parquet only carries `PULocationID`/`DOLocationID` (TLC retroactively unified all historical files, including 2015, onto the zone-ID-only schema; raw pickup/dropoff coordinates are no longer available). So instead of a lat/lon scatter, this section profiles average fare and demand by pickup zone.

In [ ]:
df_zone = pd.read_sql_query(
    """
    SELECT pickup_zone_id, zone_avg_fare, zone_trip_count
    FROM features
    GROUP BY pickup_zone_id
    ORDER BY zone_trip_count DESC
    LIMIT 20
    """,
    con,
)

print(f"Zones: {len(df_zone):,} rows")

df_zone = df_zone.sort_values("zone_trip_count")

fig, ax = plt.subplots(figsize=(10, 7))
norm = mcolors.Normalize(vmin=df_zone["zone_avg_fare"].min(), vmax=df_zone["zone_avg_fare"].max())
colors = plt.cm.plasma(norm(df_zone["zone_avg_fare"]))
bars = ax.barh(df_zone["pickup_zone_id"].astype(str), df_zone["zone_trip_count"], color=colors)
ax.set_xlabel("trip count")
ax.set_ylabel("pickup_zone_id")
ax.set_title("Top 20 Pickup Zones by Trip Volume, Colored by Avg Fare", fontsize=12)

sm = plt.cm.ScalarMappable(cmap="plasma", norm=norm)
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("zone_avg_fare ($)", fontsize=10)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap: Top 15 Features vs fare_amount

In [ ]:
numeric_cols = [
    TARGET_COL,
    "passenger_count", "trip_distance", "pickup_hour", "pickup_dow",
    "is_weekend", "is_rush_hour", "time_of_day_sin", "time_of_day_cos",
    "haversine_km", "fare_per_mile", "zone_avg_fare", "zone_trip_count",
    "avg_speed_zone",
]

df_corr = df_overview[numeric_cols].dropna()

# Pearson correlation matrix; rank by |corr with fare_amount| to pick top 15
full_corr = df_corr.corr()
top15 = (
    full_corr[TARGET_COL]
    .drop(TARGET_COL)
    .abs()
    .nlargest(14)          # 14 features + target = 15 rows/cols
    .index
    .tolist()
)
selected = [TARGET_COL] + top15
sub_corr = df_corr[selected].corr()

fig, ax = plt.subplots(figsize=(12, 5))
mask = np.triu(np.ones_like(sub_corr, dtype=bool), k=1)   # upper triangle
sns.heatmap(
    sub_corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 7},
    vmin=-1,
    vmax=1,
)
ax.set_title("Correlation Heatmap — Top 15 Features vs fare_amount", fontsize=12)
plt.tight_layout()
plt.show()

## 5. SHAP Summary Plot

In [ ]:

from IPython.display import Image, display

shap_dir = REPORTS_DIR
shap_pngs = sorted(shap_dir.glob("*shap*.png")) if shap_dir.exists() else []

if shap_pngs:
    for png in shap_pngs:
        print(f"Loading: {png.name}")
        display(Image(filename=str(png), width=900))
else:
    all_pngs = sorted(shap_dir.glob("*.png")) if shap_dir.exists() else []
    if all_pngs:
        print(f"No SHAP-named PNGs found; showing all {len(all_pngs)} figure(s) in {shap_dir}")
        for png in all_pngs:
            print(f"  {png.name}")
            display(Image(filename=str(png), width=900))
    else:
        print(
            f"No PNG files found in {shap_dir}.\n"
            "Run `make train` first to generate SHAP plots, then re-run this cell."
        )